# Interactive explorer

Companion simulator of *Network-Induced Strategic Communication in Opinion Dynamics*
([arXiv:2607.16036](https://arxiv.org/abs/2607.16036)).

The interface simulates the closed-loop dynamics

$$x_i(t+1) = \alpha_i x_i(t) + (1-\alpha_i)\sum_{j} w_{ij}\, s_j(x_j(t)),$$

where the signaling map $s_i$ is **quantized** ($q_i$, the agent's $L$-bin credible code — strategic CODA),
**linear** ($s_i(x)=x$) or **saturated** ($\operatorname{sat}(\xi_i x)$).
Options marked *beyond the paper* (tanh maps, other quantizer families, continuous time, perturbations)
are extras for exploration.

Use the **Presets** tab to load `figure4_karate_club` or `figure5_two_group` (the settings of Figs. 4 and 5)
and switch the signaling map in the **Model** tab. Set *Model profile* to *Paper assumptions* to have every
run checked against the paper's assumptions.

Requires the `ui` extra: `pip install -e ".[ui]"`.

In [ ]:
import sys
from pathlib import Path

# Use the installed package, or fall back to the repository folder.
try:
    import strategic_coda  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent))

from IPython.display import display
from strategic_coda import build_ui

ui = build_ui()
display(ui)

## Interactive opinion trajectories (Plotly)

Zoomable version of the private-opinion panel for the last run of the interface above.

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def _rgb(color) -> str:
    return f"rgb({int(color[0] * 255)}, {int(color[1] * 255)}, {int(color[2] * 255)})"


def generate_opinion_figure(_=None):
    opinion_out.clear_output(wait=True)
    with opinion_out:
        if getattr(ui, "latest_history", None) is None:
            print("Run the simulation first (Run button above).")
            return
        history, sim = ui.latest_history, ui.latest_sim
        fig = go.Figure()
        for i in range(sim.N):
            fig.add_trace(go.Scattergl(
                x=list(range(len(history))), y=history[:, i], mode="lines",
                name=f"Agent {i + 1}", line=dict(color=_rgb(sim.agent_colors[i]), width=1.5),
                opacity=0.7, showlegend=sim.N <= 15,
            ))
        fig.update_layout(
            title="Private opinions x_i(t)", xaxis_title="t", yaxis_title="x_i(t)",
            plot_bgcolor="white", hovermode="closest", height=600,
            margin=dict(l=40, r=40, t=40, b=40),
        )
        fig.update_xaxes(showgrid=True, gridcolor="LightGray")
        fig.update_yaxes(showgrid=True, gridcolor="LightGray")
        fig.show()


opinion_button = widgets.Button(description="Plot opinions", button_style="info",
                                icon="line-chart", layout=widgets.Layout(width="250px"))
opinion_button.on_click(generate_opinion_figure)
opinion_out = widgets.Output()
display(opinion_button, opinion_out)

## Interactive signaling maps (Plotly)

For the quantized map, each staircase is an agent's quantizer $q_i$: bins $[\tau_{i,\ell-1},\tau_{i,\ell})$
mapped to reconstruction levels $r_{i,\ell}$.

In [ ]:
import numpy as np


def generate_signaling_figure(_=None):
    signaling_out.clear_output(wait=True)
    with signaling_out:
        sim = getattr(ui, "latest_sim", None)
        if sim is None:
            print("Run the simulation first (Run button above).")
            return
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[-1, 1], y=[-1, 1], mode="lines", name="y = x",
                                 line=dict(color="black", width=1, dash="dash"), opacity=0.35))
        if sim.config.signal_function == "quantized":
            curves = [(np.repeat(sim.thresholds_matrix[i], 2)[1:-1],
                       np.repeat(sim.reconstruction_levels_matrix[i], 2)) for i in range(sim.N)]
            title = f"Quantizers q_i (L = {sim.L})"
        else:
            grid = np.linspace(-1.0, 1.0, 401)
            values = np.array([sim.get_public_signals(np.full(sim.N, x)) for x in grid])
            curves = [(grid, values[:, i]) for i in range(sim.N)]
            title = f"Signaling maps s_i ({sim.config.signal_function})"
        for i, (xs, ys) in enumerate(curves):
            fig.add_trace(go.Scatter(x=xs.tolist(), y=ys.tolist(), mode="lines",
                                     name=f"Agent {i + 1}",
                                     line=dict(color=_rgb(sim.agent_colors[i]), width=1.5),
                                     opacity=0.85))
        fig.update_layout(
            title=title, xaxis_title="Private opinion x", yaxis_title="Public signal s_i(x)",
            plot_bgcolor="white", hovermode="closest", height=600,
            margin=dict(l=40, r=40, t=40, b=40),
            xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]),
        )
        fig.update_xaxes(showgrid=True, gridcolor="LightGray")
        fig.update_yaxes(showgrid=True, gridcolor="LightGray")
        fig.show()


signaling_button = widgets.Button(description="Plot signaling maps", button_style="info",
                                  icon="line-chart", layout=widgets.Layout(width="250px"))
signaling_button.on_click(generate_signaling_figure)
signaling_out = widgets.Output()
display(signaling_button, signaling_out)

---
## Programmatic use

The package can be used without the interface. Export the last run of the interface:

In [ ]:
from strategic_coda.export import export_history_csv, export_run_json

export_history_csv(ui.latest_history, "run_history.csv")
export_run_json(ui.latest_sim, ui.latest_history, "run_record.json",
                bin_indices=ui.latest_bin_indices)
print("Exported run_history.csv and run_record.json")

Run the three signaling regimes of Fig. 5 from its preset:

In [ ]:
import matplotlib.pyplot as plt
from strategic_coda import StrategicCODASimulator, list_presets, load_preset, plot_simulation

print("Available presets:", list_presets())
cfg = load_preset("figure5_two_group")
for signal_function in ["linear", "saturated", "quantized"]:
    sim = StrategicCODASimulator(cfg.replace(signal_function=signal_function))
    history, _ = sim.run(cfg.steps)
    print(f"{signal_function:10s} x(T) = {history[-1].round(3)}")
    plot_simulation(sim, history, layout="horizontal", show=True)

## Animation of opinions on the network

In [ ]:
from IPython.display import HTML
from strategic_coda import SimConfig
from strategic_coda.export import animate_opinion_on_network

cfg = SimConfig(N=20, alpha=0.5, L=2, steps=50, graph_type="grid",
                initial_opinion_mode="random", opinion_seed=13)
sim = StrategicCODASimulator(cfg)
history, bin_indices = sim.run(cfg.steps, track_bins=True)

anim = animate_opinion_on_network(sim, history, interval_ms=150)
HTML(anim.to_jshtml())